In [4]:
import pymupdf

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.chat_models import init_chat_model

# rag
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loader
from langchain_community.document_loaders import PyPDFLoader

# Parser
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

# 임베딩 모델
from langchain_ollama import OllamaEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

# 벡터 스토어
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo

# selfquery
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor,EmbeddingsFilter, DocumentCompressorPipeline, CrossEncoderReranker
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever

# Reranking
from langchain_cohere import CohereRerank


In [2]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
# 모델 선언

llm = init_chat_model(model="gpt-6-luna",reasoning={"effort":"low"})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

#### 1. PDF 로드

In [7]:
loader = PyPDFLoader("./data/2018 서비스·집단 분쟁조정 사례집.pdf")
docs = loader.load()
print(len(docs))

200


#### 2. 첫번째 페이지 정보 출력
- page_content, metadata

In [10]:
print(docs[0].page_content)
print(docs[0].metadata)
print(docs[10].page_content)

소비자분쟁조정위원회
2018
서비스·집단 
분쟁조정 사례집
{'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2019-06-05T11:33:24+09:00', 'author': 'PC_A2', 'moddate': '2019-06-05T11:58:31+09:00', 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>', 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf', 'total_pages': 200, 'page': 0, 'page_label': '1'}
제1장
일
반
분
쟁
조
정
 사
례(
서
비
스)
제1장 일반분쟁조정 사례(서비스) ● 3
사
례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.
세탁 후 갑피 마모 및 경화된 가죽 
운동화에 대한 손해배상 요구
주 문
1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 
운동화, 색상 : 흰색) 1켤레를 반환한다. 
2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원
을 지급한다.
이 유
1. 기초사실
가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, 
이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. 
피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 
마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였
으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신
청인에게 손해배상(세탁비 환급 포함)을 

#### 3. 정규식을 이용한 내용 추출

In [ ]:
#사
#례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.
import re
pattern =r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."
split_text = re.findall(pattern,"".join(docs[10].page_content))
split_text

['사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.']

In [12]:
if split_text:
    parts = re.split(pattern,"".join(docs[10].page_content))
parts

['제1장\n일\n반\n분\n쟁\n조\n정\n 사\n례(\n서\n비\n스)\n제1장 일반분쟁조정 사례(서비스) ● 3\n',
 '\n세탁 후 갑피 마모 및 경화된 가죽 \n운동화에 대한 손해배상 요구\n주 문\n1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n운동화, 색상 : 흰색) 1켤레를 반환한다. \n2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n을 지급한다.\n이 유\n1. 기초사실\n가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n다는 이유로 이를 거부하였다.\n나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.\n   \n 신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만\n으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제\n인지 단정하기 어려운바, 판단 불가하다.']

In [13]:
split_text[0]

'사\n례 01 사건번호 2018일나565  | 결정일자 2018. 8. 7.'

In [ ]:
from pydantic import BaseModel, Field

class CaseMetaData(BaseModel):
    case_number:str = Field(discription="사건번호 예: 2018일나565")
    decision_date:str = Field(discription="결정일자 예: 2018. 8. 7.")

pydantic_parser = PydanticOutputParser(pydantic_object=CaseMetaData)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","다음은 분쟁조정 사례에 대한 텍스트입니다. {format_instructions}"),
        ("human","형식에 맞춰 추출해주세요 {case}"),
    ]
).partial(format_instructions=pydantic_parser.get_format_instructions())


chain = prompt | llm | pydantic_parser

response = chain.invoke({'case' : split_text[0]})
print(response)
print(response.case_number)
print(response.decision_date)

# pattern =r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

# 프롬프트
# pattern =r"\d{4}[가-힣]+\d+\d{4}\.\s*\d{1,2}\.\s*\d{1,2}\."

#2018일나565
#2018. 8. 7.

C:\Users\soldesk\AppData\Local\Temp\ipykernel_21684\882505641.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  case_number:str = Field(discription="사건번호 예: 2018일나565")
C:\Users\soldesk\AppData\Local\Temp\ipykernel_21684\882505641.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'discription'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  decision_date:str = Field(discription="결정일자 예: 2018. 8. 7.")


2018일나565
2018. 8. 7.


#### 주문 내용 추출

In [29]:
# 주 문 위치 찾기
re.search(r"주 문\n",parts[1]).span() #(38,42)

# 제목 추출

title = parts[1][:re.search(r"주 문\n",parts[1]).span()[0]].strip()
print(title)

# 내용 추출 : 주 문 부터 시작
content = parts[1][re.search(r"주 문\n",parts[1]).span()[0]:].strip()
print(content)


세탁 후 갑피 마모 및 경화된 가죽 
운동화에 대한 손해배상 요구
주 문
1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 
운동화, 색상 : 흰색) 1켤레를 반환한다. 
2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원
을 지급한다.
이 유
1. 기초사실
가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, 
이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. 
피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 
마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였
으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신
청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없
다는 이유로 이를 거부하였다.
나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.
   
 신청인이 주장하는 갑피 벗겨짐(스크래치 등) 증상은 관찰되나 현 제품 상태만
으로는 제품 훼손의 원인이 세탁 과정상 발생한 것인지 착화 환경에 따른 문제
인지 단정하기 어려운바, 판단 불가하다.


In [31]:
re.findall(r"례\s?(\d+)\s?사건번호",split_text[0])[0]

'01'

In [ ]:
pdf_docs = []
case_metadatas = {}

pattern =r"사\n례\s*\d+.*사건번호.*결정일자.*\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

for doc in docs[10:-2]:
    split_text = re.findall(pattern,"".join(doc.page_content))

    if split_text:
    
        # 사례번호
        case_metadatas['case_id'] = re.findall(r"례\s?(\d+)\s?사건번호",split_text[0])[0]

        parts = re.split(pattern,"".join(doc.page_content))
        
        if re.search(r"주 문\n",parts[1]):
            # title(metadata)
            case_metadatas['title'] = parts[1][:re.search(r"주 문\n",parts[1]).span()[0]].replace("\n","").strip()
            # page_content : 내용
            doc.page_content = parts[1][re.search(r"주 문\n",parts[1]).span()[0]:].strip()

        else: 
            # title=''
            case_metadatas['title'] = ''
            pass

        # 사례번호와 결정일자 추출 후 case_metadatas
        pydantic_parser = PydanticOutputParser(pydantic_object=CaseMetaData)

        prompt = ChatPromptTemplate.from_messages(
            [
                ("system","다음은 분쟁조정 사례에 대한 텍스트입니다. {format_instructions}"),
                ("human","형식에 맞춰 추출해주세요 {case}"),
            ]
        ).partial(format_instructions=pydantic_parser.get_format_instructions())


        chain = prompt | llm | pydantic_parser

        response = chain.invoke({'case' : case_metadatas[0]})
        case_metadatas += {response.case_number, response.decision_date}
        

    else:
        pass

_IncompleteInputError: incomplete input (2219170342.py, line 1)